In [20]:
from serpapi import SerpApiClient
from dotenv import load_dotenv
import logging
import os
import sys
import serpapi

print(sys.executable)
print(serpapi.__file__)


load_dotenv()
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


/Users/a1-6/project/hello_agent/.venv/bin/python
/Users/a1-6/project/hello_agent/.venv/lib/python3.12/site-packages/serpapi/__init__.py


In [21]:
def search(query: str) -> str:
    """
    一个基于 SerpApi 的搜索函数，返回搜索结果的摘要。
    """
    logger.info(f"正在调用 serapi 进行搜索： {query}")
    try:
        api_key = os.getenv("SERPAPI_API_KEY")
        if not api_key:
            return "SERPAPI_API_KEY 未设置，请检查 .env 文件。"
        
        params = {
            "engine": "google",
            "q": query,
            "api_key": api_key,
            "gl": "cn",
            "hl": "zh-cn"
            }
        
        client = SerpApiClient(params)
        results = client.get_dict()

        if "answer_box_list" in results:
            return "\n".join(results["answer_box_list"])
        if "answer_box" in results and "answer" in results["answer_box"]:
            return results["answer_box"]["answer"]
        if "knowledge_graph" in results and "description" in results["knowledge_graph"]:
            return results["knowledge_graph"]["description"]
        if "organic_results" in results and results["organic_results"]:
            snippets = [f"[{i+1}] {res.get('title', "")}\n{res.get("snippet", "")}" for i, res in enumerate(results["organic_results"][:3])]
            return "\n\n".join(snippets)

        return f"未找到关于{query}的相关信息。"

    except Exception as e:
        logger.error(f"搜索过程中发生错误: {e}")
        return f"搜索过程中发生错误: {e}"    




In [22]:
class ToolExecutor:
    """
    工具执行器类，用于调用不同的工具函数。
    """
    def __init__(self):
        self.tools: dict[str, dict[str, any]] = {}

    def register_tool(self, name: str, description: str, func: callable):
        """
        注册一个工具函数。
        
        """
        if name in self.tools:
            logger.warning(f"警告：工具 {name} 已经注册，将覆盖原有工具。")

        self.tools[name] = {
            "description": description,
            "function": func
        }
        logger.info(f"工具 {name} 已注册。")

    def get_tool(self, name: str) -> callable:
        """
        根据名称获取一个工具的执行函数
        """
        return self.tools.get(name, {}).get("function")
    
    def get_available_tools(self) -> str:
        """
        获取所有可用工具的格式化描述字符串
        """
        return "\n".join(f"- {name}: {info['description']}" for name, info in self.tools.items())

In [24]:
if __name__ == "__main__":
    tool_executor = ToolExecutor()

    tool_executor.register_tool(
        name="search",
        description="使用 SerpApi 进行搜索，返回搜索结果的摘要。",
        func=search
        )
    
    logger.info(f"可用工具：{tool_executor.get_available_tools()}")

    tool_name = "search"
    tool_input = "请帮我查找最新的人工智能研究论文，并总结其主要贡献"

    tool_func = tool_executor.get_tool(tool_name)
    if tool_func:
        result = tool_func(tool_input)
        logger.info(f"工具 {tool_name} 的执行结果：\n{result}")
    else:
        logger.error(f"工具 {tool_name} 未找到。")

INFO:__main__:工具 search 已注册。
INFO:__main__:可用工具：- search: 使用 SerpApi 进行搜索，返回搜索结果的摘要。
INFO:__main__:正在调用 serapi 进行搜索： 请帮我查找最新的人工智能研究论文，并总结其主要贡献
INFO:__main__:工具 search 的执行结果：
[1] AI Research - 搜索并理解论文
研究者使用AI Research 来快速找到与自己课题最相关、最新的论文。输入关键词后，他们会得到由AI 强化的摘要，突出关键发现并引用原始来源，避免幻觉。随后还能直接与论文 ...

[2] 不想自己看论文，试试这10款AI读文献神器！
1.打开网站后先上传你要读的论文，然后输入指令。 · 2.最简单的指令就是一句话，让AI帮你总结论文的内容，这样读论文最后结果会简洁一点。 请帮我读一下这篇 ...

[3] 3. 如何阅读人工智能研究论文
我将把阅读人工智能研究论文的过程分为两部分：广泛阅读和深度阅读。没有固定的先后顺序，这里有一个建议的顺序：. 针对完全没有接触过深度学习或机器学习的 ...
